# Phase 11: Scale I-JEPA Continuation

Run this notebook on a Colab GPU runtime. It tests whether the Phase 9/10 CNN-targeted gain from I-JEPA grows with higher JEPA weights or longer continuation budgets.

Default plan:

- One-epoch weight sweep: `jepa_weight` 0.05, 0.1, 0.2 at `lr=5e-5`
- Longer continuation sweep: `jepa_weight=0.05`, epochs 2 and 3 at `lr=5e-5`
- Matched DINO+MAE controls are run for each LR/epoch setting
- Seeds: 0, 1, 2
- Train limit: 1000 Imagenette train images
- Eval limit: 5000, covering full Imagenette validation
- Loss weights normalized with `--normalize-loss-weights`

The notebook uses `--skip-existing`, so rerunning after an interrupted Colab session resumes completed checkpoints and eval CSVs.

## Setup

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

assert torch.cuda.is_available(), 'Switch Colab to Runtime > Change runtime type > GPU before continuing.'

In [ ]:
%cd /content
!rm -rf jepa-transfer-attacks imagenette2-320 imagenette2-320.tgz
!git clone https://github.com/Tariolle/jepa-transfer-attacks.git
%cd /content/jepa-transfer-attacks
!pip install -q -r requirements.txt

In [ ]:
%cd /content
!wget -q --show-progress https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz
!tar -xzf imagenette2-320.tgz

from huggingface_hub import hf_hub_download

DSVA_CHECKPOINT = hf_hub_download(
    repo_id='NexusBohanLiu/dSVA',
    filename='model.pth',
    local_dir='/content/dsva_checkpoint',
)
print('dSVA checkpoint:', DSVA_CHECKPOINT)

## Configuration

In [ ]:
from pathlib import Path

REPO_ROOT = Path('/content/jepa-transfer-attacks')
TRAIN_ROOT = '/content/imagenette2-320/train'
VAL_ROOT = '/content/imagenette2-320/val'

SEEDS = [0, 1, 2]
TRAIN_LIMIT = 1000
EVAL_LIMIT = 5000
LR = '0.00005'
OUTPUT_MODE = 'scaled-delta'
OUTPUT_DIR = REPO_ROOT / 'results' / 'phase11_jepa_scale'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEIGHT_CONFIGS = ['0.05:0.00005', '0.1:0.00005', '0.2:0.00005']
LONGER_RUNS = [
    {'label': 'jw0p05_ep2', 'configs': ['0.05:0.00005'], 'epochs': 2},
    {'label': 'jw0p05_ep3', 'configs': ['0.05:0.00005'], 'epochs': 3},
]

print('Output dir:', OUTPUT_DIR)

## Optional Smoke Test

In [ ]:
import subprocess, sys

smoke_cmd = [
    sys.executable,
    'scripts/run_dsva_checkpoint_attack.py',
    '--data-root', VAL_ROOT,
    '--checkpoint', DSVA_CHECKPOINT,
    '--output-mode', 'adv',
    '--limit', '16',
    '--batch-size', '8',
    '--epsilon', '0.06274509803921569',
    '--victims', 'resnet50', 'convnext_tiny', 'vit_b_16',
    '--device', 'cuda',
    '--output-csv', 'results/phase11_smoke_released_dsva.csv',
]
print(' '.join(smoke_cmd))
subprocess.run(smoke_cmd, cwd=REPO_ROOT, check=True)

## One-Epoch JEPA Weight Sweep

In [ ]:
import subprocess, sys

for seed in SEEDS:
    seed_dir = OUTPUT_DIR / f'seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        'scripts/sweep_dsva_jepa_finetune.py',
        '--train-root', TRAIN_ROOT,
        '--val-root', VAL_ROOT,
        '--init-checkpoint', DSVA_CHECKPOINT,
        '--output-dir', str(seed_dir),
        '--run-prefix', f'phase11_seed{seed}_weight_sweep',
        '--configs', *WEIGHT_CONFIGS,
        '--limit', str(TRAIN_LIMIT),
        '--eval-limit', str(EVAL_LIMIT),
        '--epochs', '1',
        '--batch-size', '1',
        '--eval-batch-size', '8',
        '--grad-accum-steps', '8',
        '--epsilon', '0.06274509803921569',
        '--output-mode', OUTPUT_MODE,
        '--victims', 'resnet50', 'convnext_tiny', 'vit_b_16',
        '--device', 'cuda',
        '--seed', str(seed),
        '--normalize-loss-weights',
        '--skip-existing',
    ]
    print('\n' + ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)

## Longer Continuation Sweep

In [ ]:
for run in LONGER_RUNS:
    for seed in SEEDS:
        seed_dir = OUTPUT_DIR / f'seed_{seed}'
        seed_dir.mkdir(parents=True, exist_ok=True)
        cmd = [
            sys.executable,
            'scripts/sweep_dsva_jepa_finetune.py',
            '--train-root', TRAIN_ROOT,
            '--val-root', VAL_ROOT,
            '--init-checkpoint', DSVA_CHECKPOINT,
            '--output-dir', str(seed_dir),
            '--run-prefix', f"phase11_seed{seed}_{run['label']}",
            '--configs', *run['configs'],
            '--limit', str(TRAIN_LIMIT),
            '--eval-limit', str(EVAL_LIMIT),
            '--epochs', str(run['epochs']),
            '--batch-size', '1',
            '--eval-batch-size', '8',
            '--grad-accum-steps', '8',
            '--epsilon', '0.06274509803921569',
            '--output-mode', OUTPUT_MODE,
            '--victims', 'resnet50', 'convnext_tiny', 'vit_b_16',
            '--device', 'cuda',
            '--seed', str(seed),
            '--normalize-loss-weights',
            '--skip-existing',
        ]
        print('\n' + ' '.join(cmd))
        subprocess.run(cmd, cwd=REPO_ROOT, check=True)

## Aggregate Results

In [ ]:
!python scripts/analyze_phase11_scale_results.py \
  --scale-root results/phase11_jepa_scale \
  --output-detail-csv results/phase11_analysis/jepa_scale_detail.csv \
  --output-aggregate-csv results/phase11_analysis/jepa_scale_aggregate.csv

import pandas as pd
pd.read_csv('/content/jepa-transfer-attacks/results/phase11_analysis/jepa_scale_aggregate.csv')

## Package CSV Outputs

In [ ]:
!find results/phase11_jepa_scale -type f \( -name "*.csv" -o -name "*.json" \) -print > results/phase11_files_to_archive.txt
!find results/phase11_analysis -type f -name "*.csv" -print >> results/phase11_files_to_archive.txt
!printf "%s\n" results/phase11_smoke_released_dsva.csv >> results/phase11_files_to_archive.txt
!tar -czf results/phase11_csv_artifacts.tar.gz -T results/phase11_files_to_archive.txt
print('Archive:', REPO_ROOT / 'results' / 'phase11_csv_artifacts.tar.gz')